# Studi Kasus Akhir — Chatbot RAG Bahasa Indonesia dengan LLM

Notebook ini membangun aplikasi tanya-jawab berbasis dokumen. Pipeline terdiri dari knowledge base, chunking, multilingual embedding, semantic retrieval, grounded prompt, generasi LLM, evaluasi, dan antarmuka Gradio.

### Capaian

1. Menjelaskan perbedaan retrieval dan generation.
2. Mengimplementasikan top-k semantic search.
3. Menggunakan chat template model secara benar.
4. Menampilkan evidence dan menolak pertanyaan yang tidak didukung.
5. Mengevaluasi retrieval dan jawaban secara terpisah.

> Aktifkan GPU melalui **Runtime → Change runtime type → T4 GPU**.


## 1. Instalasi

Model embedding multilingual memetakan teks Indonesia ke ruang vektor. Model instruction-tuned digunakan sebagai generator. Gradio menyediakan demo web. Model publik akan diunduh saat pertama dijalankan.


In [ ]:
!pip -q install -U transformers sentence-transformers accelerate gradio pandas


## 2. Import dan konfigurasi

Generator memakai model instruction-tuned berukuran relatif kecil agar realistis untuk Colab. Parameter torch_dtype="auto" dan device_map="auto" menyesuaikan perangkat yang tersedia.


In [ ]:
import re
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TOP_K = 3

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 3. Knowledge base

Pada aplikasi nyata, teks berasal dari PDF, halaman web, atau database yang legal digunakan. Contoh menggunakan dokumen kecil agar alur dapat diperiksa. Setiap dokumen memiliki field source untuk provenance.


In [ ]:
documents = [
    {"source":"Panduan Akademik — SKS", "text":"Satuan Kredit Semester atau SKS menyatakan beban belajar mahasiswa. Mata kuliah NLP memiliki kegiatan teori, praktikum, tugas terstruktur, dan belajar mandiri."},
    {"source":"Panduan Akademik — Evaluasi", "text":"Evaluasi pembelajaran dapat mencakup kuis, ujian tengah semester, ujian akhir semester, tugas, proyek, dan presentasi sesuai rencana pembelajaran."},
    {"source":"Modul NLP — RAG", "text":"Retrieval-Augmented Generation mengambil potongan dokumen yang relevan lalu memasukkannya sebagai konteks untuk model generatif. Retrieval dan generation harus dievaluasi secara terpisah."},
    {"source":"Modul NLP — Transformer", "text":"Transformer menggunakan self-attention untuk menghubungkan token dalam urutan. Positional encoding menambahkan informasi posisi yang tidak tersedia pada attention murni."},
    {"source":"Modul NLP — Proyek", "text":"Proyek akhir NLP dikerjakan kelompok satu sampai tiga mahasiswa dan menghasilkan program serta artikel ilmiah yang mengikuti template jurnal atau prosiding."}
]

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

chunks=[]
for doc in documents:
    for i, sentence in enumerate(split_sentences(doc["text"])):
        chunks.append({"chunk_id":f'{doc["source"]}-{i}', "source":doc["source"], "text":sentence})

pd.DataFrame(chunks)


## 4. Membuat embedding index

Embedding dinormalisasi sehingga dot product ekuivalen dengan cosine similarity. Pada corpus besar, matriks dapat disimpan dalam vector database atau approximate nearest-neighbor index.


In [ ]:
embedder = SentenceTransformer(EMBED_MODEL, device=device)
corpus_embeddings = embedder.encode(
    [c["text"] for c in chunks],
    convert_to_tensor=True,
    normalize_embeddings=True
)
print("Index shape:", tuple(corpus_embeddings.shape))


## 5. Semantic retrieval

Fungsi meng-encode pertanyaan, menghitung similarity, dan mengembalikan top-k chunk bersama skor serta sumber. Threshold mencegah konteks dengan similarity sangat rendah dipakai sebagai evidence.


In [ ]:
def retrieve(question, top_k=TOP_K, min_score=0.20):
    q = embedder.encode(question, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(q, corpus_embeddings)[0]
    values, indices = torch.topk(scores, k=min(top_k, len(chunks)))
    results=[]
    for score, idx in zip(values.cpu().tolist(), indices.cpu().tolist()):
        if score >= min_score:
            results.append({**chunks[idx], "score":float(score)})
    return results

retrieve("Bagaimana RAG menghasilkan jawaban?")


## 6. Memuat LLM dan membuat grounded prompt

Chat template penting karena setiap chat model memiliki control token sendiri. System message membatasi jawaban pada konteks. Model diminta menyebut sumber dan mengakui jika evidence tidak cukup.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype="auto",
    device_map="auto"
)

SYSTEM = """Anda adalah asisten akademik berbahasa Indonesia.
Jawab hanya berdasarkan KONTEKS yang diberikan.
Jika konteks tidak cukup, katakan bahwa informasi tidak ditemukan.
Jangan membuat peraturan, angka, atau sumber baru.
Akhiri jawaban dengan daftar sumber yang digunakan."""

def build_messages(question, evidence):
    context = "\n".join(
        f'[{i+1}] Sumber: {x["source"]}\n{x["text"]}'
        for i,x in enumerate(evidence)
    ) or "Tidak ada konteks yang memenuhi ambang relevansi."
    return [
        {"role":"system", "content":SYSTEM},
        {"role":"user", "content":f"KONTEKS:\n{context}\n\nPERTANYAAN:\n{question}"}
    ]


## 7. Generasi jawaban dan provenance

Hasil fungsi generate berisi prompt dan token baru. Karena itu, token output dipotong mulai dari panjang input. Decoding dibuat hampir deterministik agar evaluasi lebih stabil.


In [ ]:
def answer_question(question, top_k=TOP_K):
    evidence = retrieve(question, top_k=top_k)
    messages = build_messages(question, evidence)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.05
        )
    new_tokens = output[0, inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return {"answer":answer, "evidence":evidence}

result = answer_question("Apa yang dimaksud dengan RAG?")
print(result["answer"])
pd.DataFrame(result["evidence"])[["source","text","score"]]


## 8. Evaluasi sederhana

Retrieval hit memeriksa apakah sumber yang diharapkan muncul pada top-k. Answer checks bukan pengganti human evaluation, tetapi berguna sebagai regression test. Tambahkan faithfulness, correctness, dan citation rubric pada proyek.


In [ ]:
tests = [
    {"question":"Apa fungsi positional encoding?", "expected_source":"Modul NLP — Transformer", "answer_keyword":"posisi"},
    {"question":"Berapa anggota kelompok proyek?", "expected_source":"Modul NLP — Proyek", "answer_keyword":"tiga"},
    {"question":"Bagaimana RAG bekerja?", "expected_source":"Modul NLP — RAG", "answer_keyword":"konteks"}
]

rows=[]
for t in tests:
    out=answer_question(t["question"])
    sources=[x["source"] for x in out["evidence"]]
    rows.append({
        "question":t["question"],
        "retrieval_hit":t["expected_source"] in sources,
        "answer_check":t["answer_keyword"].lower() in out["answer"].lower(),
        "answer":out["answer"]
    })
pd.DataFrame(rows)


## 9. Antarmuka Gradio

ChatInterface membungkus fungsi tanya-jawab menjadi aplikasi web. Tautan dengan share=True bersifat sementara dan tidak boleh digunakan untuk data rahasia.


In [ ]:
import gradio as gr

def chat(message, history):
    out = answer_question(message)
    evidence = "\n".join(
        f'- {x["source"]} (score={x["score"]:.3f})'
        for x in out["evidence"]
    )
    return f'{out["answer"]}\n\n**Evidence retrieval**\n{evidence or "Tidak ada"}'

demo = gr.ChatInterface(
    fn=chat,
    title="Asisten Akademik — RAG Bahasa Indonesia",
    description="Jawaban dibentuk dari dokumen contoh dan menampilkan evidence retrieval.",
    examples=["Apa itu RAG?", "Apa fungsi positional encoding?", "Berapa anggota kelompok proyek?"]
)
demo.launch(share=True)


## 10. Pengembangan menjadi proyek ilmiah

Eksperimen dapat membandingkan strategi chunking, sparse vs dense retrieval, top-k, embedding model, reranker, prompt, dan generator. Siapkan ground-truth relevant chunks serta reference answers. Laporkan Recall@k/MRR untuk retrieval dan correctness, faithfulness, relevance, citation accuracy, latency, serta error taxonomy untuk jawaban.

### Sumber teknis

- [Hugging Face Transformers — Chat templates](https://huggingface.co/docs/transformers/chat_templating)
- [Sentence Transformers — multilingual MiniLM model card](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
- [Gradio — ChatInterface](https://www.gradio.app/docs/gradio/chatinterface)
- [RAG paper — Lewis et al.](https://arxiv.org/abs/2005.11401)
